# LSTM Data Preparation from Shared Raw MFCC Cache


This notebook is a model-representation preparation stage. It loads the shared raw MFCC
matrices created by the analysis notebook. It does not scan audio folders, create a split, or
call Librosa MFCC extraction.


## 1. Imports and Paths


In [ ]:
# Purpose: Loads libraries for reshaping the shared MFCC matrices into one optional
# LSTM-ready sequence cache.
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def resolve_project_root():
    explicit = os.environ.get("INTRO_AI_PROJECT_ROOT")
    if explicit:
        return Path(explicit).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "Model Variants").exists():
            return candidate
        if (candidate / "Training" / "Model Variants").exists():
            return candidate / "Training"
    return Path("/content/drive/MyDrive/Colab Notebooks/Education/INM701")


PROJECT_ROOT = resolve_project_root()
SHARED_CACHE_DIR = PROJECT_ROOT / "outputs" / "shared" / "mfcc_cache"
SHARED_MANIFESTS_DIR = PROJECT_ROOT / "outputs" / "shared" / "manifests"
SHARED_CLASS_WEIGHT_PATH = PROJECT_ROOT / "outputs" / "shared" / "class_weights.json"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "lstm_optional"
CACHE_DIR = OUTPUT_DIR / "cache"
TABLES_DIR = OUTPUT_DIR / "tables"
for directory in [OUTPUT_DIR, CACHE_DIR, TABLES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required file not found: {path}. Run the analysis shared data/MFCC notebook first."
        )
    return path


## 2. Load Shared Raw MFCC Matrices


In [ ]:
# Purpose: Loads the same shared MFCC rows used by the MLP and CNN branches.
for required in [
    SHARED_CACHE_DIR / "X_train_mfcc.npy",
    SHARED_CACHE_DIR / "y_train.npy",
    SHARED_CACHE_DIR / "train_metadata.csv",
    SHARED_CACHE_DIR / "X_validation_mfcc.npy",
    SHARED_CACHE_DIR / "y_validation.npy",
    SHARED_CACHE_DIR / "validation_metadata.csv",
    SHARED_CACHE_DIR / "X_test_mfcc.npy",
    SHARED_CACHE_DIR / "y_test.npy",
    SHARED_CACHE_DIR / "test_metadata.csv",
    SHARED_CACHE_DIR / "mfcc_config.json",
    SHARED_CLASS_WEIGHT_PATH,
]:
    require_file(required)

X_train_mfcc = np.load(SHARED_CACHE_DIR / "X_train_mfcc.npy")
y_train = np.load(SHARED_CACHE_DIR / "y_train.npy")
train_metadata = pd.read_csv(SHARED_CACHE_DIR / "train_metadata.csv")
X_validation_mfcc = np.load(SHARED_CACHE_DIR / "X_validation_mfcc.npy")
y_validation = np.load(SHARED_CACHE_DIR / "y_validation.npy")
validation_metadata = pd.read_csv(SHARED_CACHE_DIR / "validation_metadata.csv")
X_test_mfcc = np.load(SHARED_CACHE_DIR / "X_test_mfcc.npy")
y_test = np.load(SHARED_CACHE_DIR / "y_test.npy")
test_metadata = pd.read_csv(SHARED_CACHE_DIR / "test_metadata.csv")
with open(SHARED_CACHE_DIR / "mfcc_config.json", "r", encoding="utf-8") as f:
    mfcc_config = json.load(f)

print("Loaded shared raw MFCC shape:", X_train_mfcc.shape[1:])
display(pd.DataFrame({"split": ["train", "validation", "test"], "rows": [len(y_train), len(y_validation), len(y_test)]}))


## 3. Convert MFCC Matrices to LSTM Sequences


In [ ]:
# Purpose: Transposes each matrix from coefficient x time into time x coefficient, so
# recurrent units receive the sequence in temporal order.
X_train_seq = np.transpose(X_train_mfcc, (0, 2, 1)).astype(np.float32)
X_validation_seq = np.transpose(X_validation_mfcc, (0, 2, 1)).astype(np.float32)
X_test_seq = np.transpose(X_test_mfcc, (0, 2, 1)).astype(np.float32)

if X_train_seq.shape[-1] != 40:
    raise RuntimeError(f"Expected 40 MFCC coefficients per time step, got {X_train_seq.shape[-1]}")

print("LSTM raw sequence shapes:", X_train_seq.shape, X_validation_seq.shape, X_test_seq.shape)


## 4. Train-Only Sequence Normalisation


In [ ]:
# Purpose: Fits coefficient-wise normalisation on training sequences only, then applies
# the same statistics to validation and test sequences.
mfcc_mean = X_train_seq.mean(axis=(0, 1), keepdims=True).astype(np.float32)
mfcc_std = X_train_seq.std(axis=(0, 1), keepdims=True).astype(np.float32)
mfcc_std = np.where(mfcc_std == 0.0, 1.0, mfcc_std).astype(np.float32)

X_train = ((X_train_seq - mfcc_mean) / mfcc_std).astype(np.float32)
X_validation = ((X_validation_seq - mfcc_mean) / mfcc_std).astype(np.float32)
X_test = ((X_test_seq - mfcc_mean) / mfcc_std).astype(np.float32)

print("Normalised LSTM shapes:", X_train.shape, X_validation.shape, X_test.shape)
print("Mean/std shapes:", mfcc_mean.shape, mfcc_std.shape)


## 5. Save One LSTM-Ready Cache


In [ ]:
# Purpose: Saves one optional LSTM-ready cache for all LSTM baseline/search/final notebooks.
np.save(CACHE_DIR / "X_train.npy", X_train)
np.save(CACHE_DIR / "X_validation.npy", X_validation)
np.save(CACHE_DIR / "X_test.npy", X_test)
np.save(CACHE_DIR / "y_train.npy", y_train.astype(np.int64))
np.save(CACHE_DIR / "y_validation.npy", y_validation.astype(np.int64))
np.save(CACHE_DIR / "y_test.npy", y_test.astype(np.int64))
np.save(CACHE_DIR / "mfcc_mean.npy", mfcc_mean)
np.save(CACHE_DIR / "mfcc_std.npy", mfcc_std)
train_metadata.to_csv(CACHE_DIR / "train_metadata.csv", index=False)
validation_metadata.to_csv(CACHE_DIR / "validation_metadata.csv", index=False)
test_metadata.to_csv(CACHE_DIR / "test_metadata.csv", index=False)
train_metadata.to_csv(TABLES_DIR / "train_manifest.csv", index=False)
validation_metadata.to_csv(TABLES_DIR / "validation_manifest.csv", index=False)
test_metadata.to_csv(TABLES_DIR / "test_manifest.csv", index=False)

feature_config = {
    "model": "LSTM",
    "optional": True,
    "source_cache": str(SHARED_CACHE_DIR),
    "canonical_manifest_source": str(SHARED_MANIFESTS_DIR),
    "shared_class_weight_path": str(SHARED_CLASS_WEIGHT_PATH),
    "source_matrix_shape": list(X_train_mfcc.shape[1:]),
    "representation": "MFCC time sequence, T x 40",
    "input_shape": list(X_train.shape[1:]),
    "normalisation": "coefficient-wise mean/std fitted on X_train_seq only",
    "mfcc_config": mfcc_config,
}
with open(CACHE_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, indent=2)

print("Saved LSTM-ready cache:", CACHE_DIR)
display(pd.DataFrame(list(feature_config.items()), columns=["key", "value"]))
